In [ ]:
import requests
import snowflake.connector
import pycountry
import os
import time
from dotenv import load_dotenv

load_dotenv("../.env")

ISO3_LIST = [
    "ZAF","CHL","SVN","IRL","LUX","PRT","SAU","UKR","LTU","KOR",
    "FRA","JPN","ITA","POL","SWE","DNK","GRC","BEL","RUS","MEX",
    "USA","CHN","DEU","CAN","NOR","ESP","NLD","AUT","FIN","HUN","ARG"
]

COUNTRIES = []
for iso3 in ISO3_LIST:
    c = pycountry.countries.get(alpha_3=iso3)
    if c:
        COUNTRIES.append((iso3, c.alpha_2))
    else:
        print(f"NO ENCONTRADO: {iso3}")

YEARS = [2019, 2020, 2021, 2022, 2023]
TOP_N = 200
CONCEPT_ID = "C154945302"

def fetch_top_papers(iso3, iso2, year):
    url = "https://api.openalex.org/works"
    params = {
        "filter": f"concepts.id:{CONCEPT_ID},from_publication_date:{year}-01-01,to_publication_date:{year}-12-31,authorships.countries:{iso2}",
        "sort": "cited_by_count:desc",
        "per_page": TOP_N,
        "select": "id,cited_by_count",
        "mailto": "tfg@project.com"
    }
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        print(f"ERROR {iso3} {year}: {r.status_code}")
        return None
    results = r.json().get("results", [])
    if not results:
        print(f"SKIP {iso3} {year}: 0 resultados")
        return None
    total_cited = sum(p.get("cited_by_count", 0) for p in results)
    paper_count = len(results)
    mcp = round(total_cited / paper_count, 4)
    return (iso3, year, total_cited, paper_count, mcp)

conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse="COMPUTE_WH",
    database="IA_PAISES",
    schema="RAW"
)
cur = conn.cursor()

cur.execute("USE ROLE ACCOUNTADMIN")
cur.execute("USE DATABASE IA_PAISES")
cur.execute("""
    CREATE TABLE IF NOT EXISTS RAW.OPENALEX_CITATIONS (
        COUNTRY_CODE VARCHAR(3),
        YEAR INT,
        TOTAL_CITED_BY INT,
        PAPER_COUNT INT,
        MEAN_CITATIONS_PER_PAPER FLOAT
    )
""")
cur.execute("TRUNCATE TABLE RAW.OPENALEX_CITATIONS")

rows = []
for iso3, iso2 in COUNTRIES:
    for year in YEARS:
        result = fetch_top_papers(iso3, iso2, year)
        if result:
            rows.append(result)
            print(f"OK {iso3} {year} → MCP={result[4]}")
        time.sleep(0.5)

if rows:
    cur.executemany(
        "INSERT INTO RAW.OPENALEX_CITATIONS VALUES (%s, %s, %s, %s, %s)",
        rows
    )
    conn.commit()
    print(f"\nCargadas {len(rows)} filas en RAW.OPENALEX_CITATIONS")
else:
    raise Exception("0 filas cargadas — revisar API o credenciales")

cur.execute("SELECT COUNT(*) FROM RAW.OPENALEX_CITATIONS")
print(f"Verificación Snowflake: {cur.fetchone()[0]} filas")

cur.close()
conn.close()